In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Running this notebook on: ", device)

import spateo as st
print("Last run with spateo version:", st.__version__)

# Other imports
import warnings, string
import anndata as ad
import dynamo as dyn
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Uncomment the following if running on the server
import pyvista as pv
pv.start_xvfb()
from matplotlib.colors import ListedColormap, rgb2hex
sns.set_theme(context="paper", style="ticks", font_scale=1)
warnings.filterwarnings('ignore')
# %load_ext autoreload
# %autoreload 2
from tqdm import tqdm

Running this notebook on:  cpu


2025-11-21 00:54:27.556821: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/root/miniforge3/envs/spateo/lib/python3.8/site-packages/geopandas/_compat.py:124: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


fastpd is not installed. If you need mesh correction, please compile the fastpd library.
Last run with spateo version: 0.0.0


In [ ]:
import scanpy as sc
names = [
    '20_B03606F3G5_WT202405020032.h5ad',
 '22_B03606C4E6_WT202403310050.h5ad',
 '23_B03609A4D6_WT202404150263.h5ad',
 '27_B03610C1E3_WT202403310051.h5ad',
 '31_B03619A1D3_WT202403310052.h5ad',
 '35_B03619E4G6_WT202403310053.h5ad',
 '39_A03589A1D4_WT202403310046.h5ad',
 '43_A03590E1G4_WT202403310064.h5ad',
 '47_A03593C1F3_WT202403310068.h5ad',
 '51_B03605C2E5_WT202406020126.h5ad',
 '55_B03613E3G6_WT202403310069.h5ad',
 '59_B03612E4G6_WT202403310059.h5ad',
 '63_B03606C1E3_WT202403310061.h5ad',
 '67_A03595A1D3_WT202403310062.h5ad',
 '71_A03595A4D6_WT202403310063.h5ad',
    '75_D03468D1E3_WT202403310066.h5ad',
    '80_D03473D4E6_WT202403310070.h5ad',
    '84_B03423D1E3_WT202403310065.h5ad',
]

adata_all = sc.read_h5ad('/data/work/05.cluster/FuseMap/20250106/mid_hind_latent_embeddings_all_single_pretrain/dmt_leiden_20250108_1.h5ad')
adata_bound_list = []
for name in tqdm(names):
    adata = adata_all[adata_all.obs['slice_code'] == name].copy()
    adata.obs_names_make_unique()
    if name == '47_A03593C1F3_WT202403310068.h5ad':
        adata.obsm['align_spatial_3d'] = np.array([adata.obsm['align_spatial_2d'][:,0],
                                                   adata.obsm['align_spatial_2d'][:,1],
                                                   [1187.5 for i in range(len(adata))]
                                                   ]).T
    adata = adata[adata.obs.sample(frac = 0.01, random_state = 42, replace = True).index].copy()
    adata_bound_list.append(adata)
adata = ad.concat(adata_bound_list)
adata.obs['region_h1'] = 'mhb'
color_dict = {'mhb': '#ffdddd'}
pc, plot_cmap = st.tdr.construct_pc(adata=adata.copy(), spatial_key="align_spatial_3d", groupby="region_h1", key_added="tissue", colormap=color_dict)
mesh, _, _ = st.tdr.construct_surface(pc=pc, key_added="tissue", 
                                      alpha=0.6, cs_method="marching_cube", cs_args={"mc_scale_factor": 1.02}, 
                                      smooth=5000, scale_factor=1.08)
mesh.save('/data/work/12.pl_ply/mesh/mhb.obj', texture = mesh['tissue_rgba'][:,:3])

In [2]:
adata_all = sc.read_h5ad('/data/work/05.cluster/FuseMap/20250106/thalamus_latent_embeddings_all_spatial_pretrain/dmt_leiden_20250108_1.h5ad')
names = [
 '14_A03591A1C3_WT202403310045.h5ad',
 '16_A03592A4C6_WT202403310044.h5ad',
 '18_B03602C4D6_WT202405020031.h5ad',
 '20_B03606F3G5_WT202405020032.h5ad',
 '22_B03606C4E6_WT202403310050.h5ad',
 '23_B03609A4D6_WT202404150263.h5ad',
 '27_B03610C1E3_WT202403310051.h5ad',
 '31_B03619A1D3_WT202403310052.h5ad',
 '35_B03619E4G6_WT202403310053.h5ad',
 '39_A03589A1D4_WT202403310046.h5ad',
 '43_A03590E1G4_WT202403310064.h5ad',
 '47_A03593C1F3_WT202403310068.h5ad',
 '51_B03605C2E5_WT202406020126.h5ad',
 '55_B03613E3G6_WT202403310069.h5ad',
 '59_B03612E4G6_WT202403310059.h5ad',
 '63_B03606C1E3_WT202403310061.h5ad',
 '67_A03595A1D3_WT202403310062.h5ad',
 '71_A03595A4D6_WT202403310063.h5ad',
 '75_D03468D1E3_WT202403310066.h5ad',
 '80_D03473D4E6_WT202403310070.h5ad',
 '84_B03423D1E3_WT202403310065.h5ad',
 '89_D03466A1C3_WT202403310058.h5ad',
 '99_D03470D1E3_WT202404290071.h5ad',
 '104_B03615F3G5_WT202405020035.h5ad',
 '105_D03468A4C6_WT202403310067.h5ad',
]
adata_bound_list = []
for name in tqdm(names):
    adata = adata_all[adata_all.obs['slice_code'] == name].copy()
    adata.obs_names_make_unique()
    if name == '47_A03593C1F3_WT202403310068.h5ad':
        adata.obsm['align_spatial_3d'] = np.array([adata.obsm['align_spatial_2d'][:,0],
                                                   adata.obsm['align_spatial_2d'][:,1],
                                                   [1187.5 for i in range(len(adata))]
                                                   ]).T
    adata = adata[adata.obs.sample(frac = 0.01, random_state = 42, replace = False).index].copy()
    adata_bound_list.append(adata)
adata = ad.concat(adata_bound_list)
adata.obs['region_h1'] = 'thalamus'
color_dict = {'thalamus': '#6361f1'}
pc, plot_cmap = st.tdr.construct_pc(adata=adata.copy(), spatial_key="align_spatial_3d", groupby="region_h1", key_added="tissue", colormap=color_dict)
mesh, _, _ = st.tdr.construct_surface(pc=pc, key_added="tissue", 
                                      alpha=0.6, cs_method="marching_cube", cs_args={"mc_scale_factor": 1.02}, 
                                      smooth=5000, scale_factor=1.15)
mesh.save('/data/work/12.pl_ply/mesh/thalamus.obj', texture = mesh['tissue_rgba'][:,:3])

100%|██████████| 25/25 [00:12<00:00,  2.02it/s]


In [2]:
csv = pd.read_csv('/data/work/0521.csv', index_col = 'Unnamed: 0')
names = ['15_C03627F5_WT202403180043.h5ad',
         '17_C03627F6_WT202403270557.h5ad',
'19_D03657F1_WT202403110530.h5ad',
'21_D03657F2_WT202403110531.h5ad',
'22_B03606C4E6_WT202403310050.h5ad',
'23_B03609A4D6_WT202404150263.h5ad',
'27_B03610C1E3_WT202403310051.h5ad',
'31_B03619A1D3_WT202403310052.h5ad',
'35_B03619E4G6_WT202403310053.h5ad',
'39_A03589A1D4_WT202403310046.h5ad',
'43_A03590E1G4_WT202403310064.h5ad',
'47_A03593C1F3_WT202403310068.h5ad',
'51_B03605C2E5_WT202406020126.h5ad',
'55_B03613E3G6_WT202403310069.h5ad',
'59_B03612E4G6_WT202403310059.h5ad',
'63_B03606C1E3_WT202403310061.h5ad',
'67_A03595A1D3_WT202403310062.h5ad',
'71_A03595A4D6_WT202403310063.h5ad',
'76_D03656A5_WT202403280404.h5ad',
'81_D03657C6_WT202403110520.h5ad',
'85_B03611D2_WT202403110546.h5ad',
'90_A03592D3_WT202403110532.h5ad',
'95_B03602D1_WT202403110535.h5ad',
'100_B03609G1_WT202403280406.h5ad',
]
adata = sc.read_h5ad('/data/work/05.cluster/FuseMap/20250106/cerebellum_latent_embeddings_all_single_pretrain/dmt_leiden_20250108_1.h5ad')
adata.obs_names_make_unique()
adata = adata[adata.obs['slice_code'].isin(names)].copy()
adatas = []
for name in names:
    temp = adata[adata.obs['slice_code'] == name]
    slice_code = name.replace('.h5ad', '')
    idx, chip, task = slice_code.split('_')
    z = csv[csv['chip'] == chip]['z'].values[0]
    temp.obsm['align_spatial_3d'] = np.array([temp.obsm['align_spatial_2d'][:,0],
                                              temp.obsm['align_spatial_2d'][:,1],
                                              [z for i in range(len(temp))]
                                              ]).T
    temp = temp[temp.obs.sample(frac = 0.01, random_state = 42, replace = False).index].copy()
    adatas.append(temp)
adata = ad.concat(adatas)

In [3]:
adata1 = sc.read_h5ad('/data/work/05.cluster/FuseMap/20251114/adata_20251119.h5ad')
adata1.obs_names_make_unique()

In [4]:
adata1

AnnData object with n_obs × n_vars = 347620 × 27847
    obs: 'orig.ident', 'x', 'y', 'region_h2'
    obsm: 'align_spatial_2d', 'align_spatial_3d', 'spatial', 'spatial_division'

In [5]:
adata1 = adata1[adata1.obs.sample(frac = 0.1, random_state = 42, replace = False).index].copy()

In [6]:
adata2 = ad.concat([adata, adata1])

In [7]:
adata2

AnnData object with n_obs × n_vars = 52459 × 26923
    obs: 'orig.ident', 'x', 'y', 'region_h2'
    obsm: 'align_spatial_2d', 'align_spatial_3d', 'spatial', 'spatial_division'

In [9]:
adata2.obs['region_h1'] = 'brain'
color_dict = {'brain': '#BDBDBD'}
pc, plot_cmap = st.tdr.construct_pc(adata=adata2.copy(), spatial_key="align_spatial_3d", groupby="region_h1", key_added="tissue", colormap=color_dict)
mesh, _, _ = st.tdr.construct_surface(pc=pc, key_added="tissue", 
                                      alpha=0.6, cs_method="marching_cube", cs_args={"mc_scale_factor": 1.02}, 
                                      smooth=5000, scale_factor=1.08)
mesh.save('/data/work/12.pl_ply/mesh/brain.obj', texture = mesh['tissue_rgba'][:,:3])